# Leaf Recognition Program
Train a CNN to classify mango and jackfruit leaf images using the prepared train/validation/test split.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
dataset_root = Path(r"c:\Users\alvee\Desktop\AI Lab\Leaf_Recognition\split_dataset")
image_size = (180, 180)
batch_size = 32
epochs = 20
model_output = Path(r"c:\Users\alvee\Desktop\AI Lab\Leaf_Recognition\leaf_recognition_model.keras")

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_root / "train",
    label_mode="int",
    image_size=image_size,
    batch_size=batch_size,
    shuffle=True,
    seed=42,
)
validation_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_root / "validation",
    label_mode="int",
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_root / "test",
    label_mode="int",
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False,
)

class_names = train_ds.class_names
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).cache().prefetch(buffer_size=AUTOTUNE)
validation_ds = validation_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="augmentation")

model = models.Sequential([
    layers.Input(shape=(image_size[0], image_size[1], 3)),
    data_augmentation,
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Dropout(0.3),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=epochs,
    callbacks=[early_stopping, reduce_lr],
)

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

model.save(model_output)
print(f"Saved model to: {model_output}")